In [81]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN, KMeans, AgglomerativeClustering, MiniBatchKMeans, Birch, SpectralClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.utils import resample
from scipy.cluster.hierarchy import linkage, dendrogram
import faiss
import time
import hdbscan
from sklearn.base import clone
import numpy as np
import itertools

In [82]:
def plot_clusters_pca(X, labels, title):
    X = np.array(X)
    if X.ndim == 1:
        raise ValueError("Input data X must be 2D for plotting clusters.")
    if X.shape[1] < 2:
        raise ValueError("Input data X must have at least two features (columns) for 2D plotting.")
    print(title,"Silhouette:", silhouette_score(X, labels) if len(set(labels)) > 1 else "N/A")
    plt.figure(figsize=(6, 5))
    plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', s=10)
    plt.title(title)
    plt.grid(True)
    plt.show()

In [83]:
def plot_clusters(X, labels, title):
    X = np.array(X)
    if X.ndim == 1:
        raise ValueError("Input data X must be 2D for plotting clusters.")
    if X.shape[1] < 2:
        raise ValueError("Input data X must have at least two features (columns) for 2D plotting.")
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    print(title,"Silhouette:", silhouette_score(X, labels) if len(set(labels)) > 1 else "N/A")
    plt.figure(figsize=(6, 5))
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=10)
    plt.title(title)
    plt.grid(True)
    plt.show()

In [84]:
def metrike(title,X,labels):
    print("Silhouette +",title,":", silhouette_score(X, labels))
    print("Davies-Bouldin indeks: +",title,":", davies_bouldin_score(X, labels))

In [85]:
def metrike1(iter,X,labels):
    return {
        "iter:": iter,
        "Silhouette": round(silhouette_score(X, labels), 3),
        "Davies-Bouldin indeks": round(davies_bouldin_score(X, labels), 3),
    }

In [86]:
def toExel(exel, title):
    df_exel = pd.DataFrame(exel)
    df_exel.to_excel(f"klasterCLICK2/{title}.xlsx", index=False)

In [87]:
def dict_product(param_grid):
    keys = param_grid.keys()
    values = param_grid.values()
    for instance in itertools.product(*values):
        yield dict(zip(keys, instance))

In [88]:
def cluster11(model, title, X, y, params):

    best_score = -np.inf
    best_model = None
    best_params = None
    best_metrics = None

    for param_set in dict_product(params):

        m = clone(model)
        m.set_params(**param_set)

        m.fit(X)
        labels = m.labels_

        met = metrike1(title, X, labels)

        score = met["Silhouette"]  

        if score > best_score:
            best_score = score
            best_model = m
            best_params = param_set
            best_metrics = met
        print("gotov")
    print("Najbolji parametri:", best_params)
    print(best_metrics)
    row = {**best_params, **met}
    print("*"*50)
    return row

In [89]:
def cluster22(model, title, X, i, params=None):

    # GRID SEARCH
    best_score = -np.inf
    best_model = None
    best_params = None
    best_metrics = None

    for param_set in dict_product(params):

        m = clone(model)
        m.set_params(**param_set)

        labels = m.fit_predict(X)
        met = metrike1(title, X, labels)


        score = met["Silhouette"] 

        if score > best_score:
            best_score = score
            best_model = m
            best_params = param_set
            best_metrics = met
        print("gotov!")

    print("Najbolji parametri:", best_params)
    print(best_metrics)
    row = {**best_params, **met}
    print("*"*50)
    return row

In [90]:
def faisscluster1(title, X, i, params=None):

    X = np.ascontiguousarray(X.astype('float32'))
    d = X.shape[1]

    best_score = -np.inf
    best_model = None
    best_params = None
    best_metrics = None

    for param_set in dict_product(params):

        niter = param_set.get("niter")
        nredo = param_set.get("nredo")
        seed = param_set.get("seed")

        faiss_kmeans = faiss.Kmeans(d=d,k=i,niter=niter,nredo=nredo,seed=123,verbose=False)

        faiss_kmeans.train(X)
        D, I = faiss_kmeans.index.search(X, 1)
        labels = I.flatten()

        met = metrike1(title, X, labels)

        score = met["Silhouette"]   # koristi tačan ključ

        if score > best_score:
            best_score = score
            best_model = faiss_kmeans
            best_params = param_set
            best_metrics = met
        print("gotov")
    print("Najbolji parametri:", best_params)
    print(best_metrics)
    row = {**best_params, **met}
    print("*"*50)

    return row

In [91]:
def mode_func(x):
    return x.mode().iloc[0] if not x.mode().empty else x.iloc[0]

In [92]:
df = pd.read_csv(r'podaci\clickstream+data+for+online+shopping\e-shop clothing 2008 preprocessed.csv', encoding='cp1252', sep=',')
print(df.columns)

Index(['year', 'month', 'day', 'order', 'country', 'session ID',
       'main category', 'clothing model', 'colour', 'location',
       'model photography', 'price', 'price 2', 'page'],
      dtype='object')


In [93]:
session_df = df.groupby("session ID").agg({
    "month": mode_func,
    "day": mode_func,
    "order": "count",   
    "page": ["nunique", "max"],
    "country": mode_func,
    "main category": [mode_func, "nunique"],
    "clothing model": [mode_func, "nunique"],
    "colour": [mode_func, "nunique"],
    "location": [mode_func, "nunique"],
    "model photography": [mode_func, "nunique"],
    "price": ["mean", "max", "min"],
    "price 2": ["mean", "max", "min"]
})

In [94]:
session_df.columns = ["_".join(col) for col in session_df.columns]
session_df = session_df.reset_index()

In [95]:
session_df.head()

,session ID,month_mode_func,day_mode_func,order_count,page_nunique,page_max,country_mode_func,main category_mode_func,main category_nunique,clothing model_mode_func,...,location_mode_func,location_nunique,model photography_mode_func,model photography_nunique,price_mean,price_max,price_min,price 2_mean,price 2_max,price 2_min
0,1,4,1,9,3,5,29,2,4,4,...,1,6,2,2,42.111111,57,28,1.444444,2,1
1,2,4,1,10,2,2,29,2,3,1,...,1,6,1,2,50.000000,67,38,1.200000,2,1
2,3,4,1,6,2,5,21,3,3,51,...,2,4,1,2,42.166667,48,28,1.333333,2,1
3,4,4,1,4,3,3,21,1,2,27,...,1,4,1,1,45.250000,62,33,1.500000,2,1
4,5,4,1,1,1,2,9,3,1,89,...,1,1,1,1,57.000000,57,57,1.000000,1,1


In [96]:
X = session_df.drop("session ID", axis=1)

In [97]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

KLASTERI

In [98]:
exel=[]
params_minibatch = {
    "batch_size": [16,32,64,128,256]
}

In [99]:
for i in range(2,6):
    mb_kmeans = MiniBatchKMeans(n_clusters=i, random_state=123)
    
    print("---------------- Broj klastera ",i,"--------------------")
    
    metrika = cluster11(mb_kmeans,"MiniBatchKMeans_X",X,i,params_minibatch)
    # metrika = cluster11(mb_kmeans,"MiniBatchKMeans_X_pca",X_pca,i,params_minibatch)
    exel.append(metrika)
toExel(exel,'MiniBatchKMeans_X')

---------------- Broj klastera  2 --------------------
gotov
gotov
gotov
gotov
gotov
Najbolji parametri: {'batch_size': 64}
{'iter:': 'MiniBatchKMeans_X', 'Silhouette': np.float64(0.527), 'Davies-Bouldin indeks': np.float64(0.702)}
**************************************************
---------------- Broj klastera  3 --------------------
gotov
gotov
gotov
gotov
gotov
Najbolji parametri: {'batch_size': 64}
{'iter:': 'MiniBatchKMeans_X', 'Silhouette': np.float64(0.377), 'Davies-Bouldin indeks': np.float64(0.952)}
**************************************************
---------------- Broj klastera  4 --------------------
gotov
gotov
gotov
gotov
gotov
Najbolji parametri: {'batch_size': 32}
{'iter:': 'MiniBatchKMeans_X', 'Silhouette': np.float64(0.312), 'Davies-Bouldin indeks': np.float64(1.138)}
**************************************************
---------------- Broj klastera  5 --------------------
gotov
gotov
gotov
gotov
gotov
Najbolji parametri: {'batch_size': 128}
{'iter:': 'MiniBatchKMeans

In [100]:
exel=[]
params_kmeans = {
    "init": ["k-means++", "random"],
    "n_init": [3,5,10]
}

In [101]:
for i in range(2,6):
    kmeans = KMeans(n_clusters=i , random_state=123)
    
    print("---------------- Broj klastera ",i,"--------------------")

    metrika = cluster22(kmeans,"KMeans_X",X,i,params_kmeans)
    # metrika = cluster22(kmeans,"KMeans_X_pca",X_pca,i,params_kmeans)
    exel.append(metrika)
toExel(exel,'KMeans_X')

---------------- Broj klastera  2 --------------------
gotov!
gotov!
gotov!
gotov!
gotov!
gotov!
Najbolji parametri: {'init': 'k-means++', 'n_init': 3}
{'iter:': 'KMeans_X', 'Silhouette': np.float64(0.515), 'Davies-Bouldin indeks': np.float64(0.741)}
**************************************************
---------------- Broj klastera  3 --------------------
gotov!
gotov!
gotov!
gotov!
gotov!
gotov!
Najbolji parametri: {'init': 'k-means++', 'n_init': 3}
{'iter:': 'KMeans_X', 'Silhouette': np.float64(0.377), 'Davies-Bouldin indeks': np.float64(0.953)}
**************************************************
---------------- Broj klastera  4 --------------------
gotov!
gotov!
gotov!
gotov!
gotov!
gotov!
Najbolji parametri: {'init': 'k-means++', 'n_init': 3}
{'iter:': 'KMeans_X', 'Silhouette': np.float64(0.312), 'Davies-Bouldin indeks': np.float64(1.138)}
**************************************************
---------------- Broj klastera  5 --------------------
gotov!
gotov!
gotov!
gotov!
gotov!
goto

In [102]:
exel = []
params_faiss = {
    "niter": [20,50,100],
    "nredo": [1,5,10]
}

In [103]:
for i in range(2,6):
    # metrika = faisscluster1("Faiss_X", X, i,params_faiss)
    metrika = faisscluster1("Faiss_X_pca", X_pca, i,params_faiss)
    exel.append(metrika)
toExel(exel,'Faiss_X_pca')

gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
Najbolji parametri: {'niter': 20, 'nredo': 5}
{'iter:': 'Faiss_X_pca', 'Silhouette': np.float32(0.594), 'Davies-Bouldin indeks': np.float64(0.611)}
**************************************************
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
Najbolji parametri: {'niter': 20, 'nredo': 1}
{'iter:': 'Faiss_X_pca', 'Silhouette': np.float32(0.512), 'Davies-Bouldin indeks': np.float64(0.657)}
**************************************************
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
Najbolji parametri: {'niter': 20, 'nredo': 1}
{'iter:': 'Faiss_X_pca', 'Silhouette': np.float32(0.472), 'Davies-Bouldin indeks': np.float64(0.731)}
**************************************************
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
Najbolji parametri: {'niter': 20, 'nredo': 10}
{'iter:': 'Faiss_X_pca', 'Silhouette': np.float32(0.476), 'Davies-Bouldin indeks': np.float64(0.755)}
**************************************

In [ ]:
exel = []
params_spectral = {
    "affinity": ["nearest_neighbors"],
    "gamma": [0.1,1,10],
    "n_neighbors": [5,10,20],
    "assign_labels": ["kmeans", "discretize"]
}

In [105]:
for i in range(2,6):
    spectral = SpectralClustering(n_clusters=i, random_state=123)

    print("---------------- Broj klastera ",i,"--------------------")

    # metrika = cluster22(spectral,"Spectral_X",X,i,params_spectral)
    metrika = cluster22(spectral,"Spectral_X_PCA",X_pca, i,params_spectral)
    exel.append(metrika)
toExel(exel,'Spectral_X_PCA')

---------------- Broj klastera  2 --------------------


c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


gotov!


c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


gotov!


c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


KeyboardInterrupt: 

In [106]:
exel = []
params_agg = {
    "linkage": ["ward","complete","average"],
    "metric":["euclidean"]
}

In [107]:
for i in range(2,6):
    agg = AgglomerativeClustering(n_clusters=i)

    print("---------------- Broj klastera ",i,"--------------------")

    # metrika = cluster22(agg,"AGG_X",X,i,params_agg)
    metrika = cluster22(agg,"AGG_X_PCA",X_pca,i,params_agg)
    exel.append(metrika)
toExel(exel,'AGG_X_PCA')

---------------- Broj klastera  2 --------------------
gotov!
gotov!
gotov!
Najbolji parametri: {'linkage': 'average', 'metric': 'euclidean'}
{'iter:': 'AGG_X_PCA', 'Silhouette': np.float64(0.649), 'Davies-Bouldin indeks': np.float64(0.443)}
**************************************************
---------------- Broj klastera  3 --------------------
gotov!
gotov!
gotov!
Najbolji parametri: {'linkage': 'average', 'metric': 'euclidean'}
{'iter:': 'AGG_X_PCA', 'Silhouette': np.float64(0.619), 'Davies-Bouldin indeks': np.float64(0.467)}
**************************************************
---------------- Broj klastera  4 --------------------
gotov!
gotov!
gotov!
Najbolji parametri: {'linkage': 'average', 'metric': 'euclidean'}
{'iter:': 'AGG_X_PCA', 'Silhouette': np.float64(0.597), 'Davies-Bouldin indeks': np.float64(0.602)}
**************************************************
---------------- Broj klastera  5 --------------------
gotov!
gotov!
gotov!
Najbolji parametri: {'linkage': 'average', 'm

In [108]:
exel = []
params_birch = {
    "threshold": [0.01,0.1,0.3,0.5,1.0],
    "branching_factor": [25,50,100],
}

In [109]:
for i in range(2,6):
    birch = Birch(n_clusters=i)
    
    print("---------------- Broj klastera ",i,"--------------------")
    
    # metrika = cluster11(birch,"Birch_X",X,i,params_birch)
    metrika = cluster11(birch,"Birch_X_pca",X_pca,i,params_birch)
    exel.append(metrika)
toExel(exel,'Birch_X_pca')

---------------- Broj klastera  2 --------------------
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
Najbolji parametri: {'threshold': 0.1, 'branching_factor': 100}
{'iter:': 'Birch_X_pca', 'Silhouette': np.float64(0.621), 'Davies-Bouldin indeks': np.float64(0.51)}
**************************************************
---------------- Broj klastera  3 --------------------
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
Najbolji parametri: {'threshold': 1.0, 'branching_factor': 50}
{'iter:': 'Birch_X_pca', 'Silhouette': np.float64(0.533), 'Davies-Bouldin indeks': np.float64(0.663)}
**************************************************
---------------- Broj klastera  4 --------------------
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
gotov
Najbolji parametri: {'threshold': 0.01, 'branching_factor': 50}
{'iter:': 'Birch_X_pca', 'Silhouette': np.float64(0.484), 'Davies-Bou

In [ ]:
linked = linkage(X_pca, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(linked, orientation='top', distance_sort='descending', show_leaf_counts=False, no_labels=True)
plt.show()